# Assignment Runbook: Qwen2.5-1.5B Fine-Tuning

This notebook is written as a student workflow for Assignment 2.

## What this notebook demonstrates
- Data collection and preparation for financial-assistant tasks (EN/VI/ZH)
- Local fine-tuning of a pretrained model with LoRA/QLoRA
- Before-vs-after comparison using measurable metrics
- Cross-model and cross-language evidence for report justification

## Expected assignment evidence
By the end of the notebook, you should have:
1. Clean train/eval JSONL files
2. Fine-tuned adapter checkpoint
3. Metric tables (base vs fine-tuned, cross-model, cross-lingual)
4. Reproducibility manifest for report appendix

## Section flow
1. Setup and environment check
2. Data processing and validation
3. Baseline model evaluation
4. Fine-tuning
5. Post-training evaluation
6. Model comparison
7. Multilingual analysis
8. Optional deployment note
9. Reproducibility output

## Section 1 — Setup and environment check

Use this section to confirm your runtime is ready before training.

Checklist:
- Install required packages
- (Optional) provide `HF_TOKEN` for higher download reliability
- Detect compute device (`cuda`, `mps`, or `cpu`)
- Record environment info for your report

In [ ]:
!pip uninstall -y torchao torch torchvision torchaudio
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!pip install \
  transformers==4.44.0 \
  datasets==2.21.0 \
  peft==0.13.2 \
  trl==0.11.4 \
  bitsandbytes==0.43.3 \
  accelerate==0.34.2 \
  numpy==1.26.4 \
  scikit-learn \
  ipywidgets

Found existing installation: torch 2.0.1+cu118
Uninstalling torch-2.0.1+cu118:
  Successfully uninstalled torch-2.0.1+cu118
Found existing installation: torchvision 0.15.2+cu118
Uninstalling torchvision-0.15.2+cu118:
  Successfully uninstalled torchvision-0.15.2+cu118
Found existing installation: torchaudio 2.0.2+cu118
Uninstalling torchaudio-2.0.2+cu118:
  Successfully uninstalled torchaudio-2.0.2+cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.0.1%2Bcu118-cp311-cp311-linux_x86_64.whl (2267.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.15.2%2Bcu118-cp311-cp311-linux_x86_64.whl (6.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.0.2%2Bcu118-cp311-cp311-linux_x86_64.whl (4.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]3 [torchaudio]]
ERROR: Invalid requirement: '\\': Expected package name at the start of dependency specifier
    

In [2]:
# Reproducibility configuration for PyTorch-based training
SEED = 42

# 2) Deterministic seeds and backend flags
import os, random, platform, hashlib
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch, "use_deterministic_algorithms"):
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

# 3) Environment visibility for report/debugging
HAS_CUDA = torch.cuda.is_available()
HAS_MPS = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
DEVICE = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
USE_4BIT = DEVICE == "cuda"

def _rng_fingerprint() -> str:
    b = torch.get_rng_state().numpy().tobytes()
    return hashlib.sha256(b).hexdigest()[:16]

print("=== Reproducible bootstrap configured ===")
print(f"SEED={SEED}")
print(f"python={platform.python_version()} torch={torch.__version__}")
print(f"device={DEVICE} use_4bit={USE_4BIT}")
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA: {torch.cuda.get_device_name(0)} ({props.total_memory / 1e9:.1f} GB VRAM)")
print(f"torch_rng_fingerprint={_rng_fingerprint()}")

=== Reproducible bootstrap configured ===
SEED=42
python=3.11.13 torch=2.0.1+cu118
device=cuda use_4bit=True
CUDA: Tesla V100-PCIE-32GB (34.1 GB VRAM)
torch_rng_fingerprint=5feb54a46230d321


In [3]:
import os, platform
from pathlib import Path
from dotenv import load_dotenv

# Colab-friendly root
PROJECT_ROOT = Path(os.getenv("REPO_DIR", os.getcwd())).resolve()
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")  # optional for public models

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_NAME = "lora-qwen25-1p5b-finbot-v2"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = PROJECT_ROOT / "raw_data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / ADAPTER_NAME

SAFETY_UNSAFE_TERMS = ["guaranteed profit", "risk-free", "cannot lose", "guaranteed return"]
VALID_TASK_MAP = {"planning": "PLANNING", "investment": "INVESTMENT", "trading": "TRADING"}
VALID_LANG_CODES = {"en", "vi", "zh"}

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

TRAIN_BATCH_SIZE = 4
TRAIN_GRAD_ACC_STEPS = 2
TRAIN_LR = 2e-4
TRAIN_EPOCHS = 1
TRAIN_LOGGING_STEPS = 10
TRAIN_MAX_SEQ_LENGTH = 2048
TRAIN_SEED = 42
TRAIN_PACKING = True
TRAIN_DATALOADER_WORKERS = 2
TRAIN_SAVE_STRATEGY = "no"
TRAIN_REPORT_TO = "none"

GEN_MAX_NEW_TOKENS = 700

for _d in (DATA_DIR, RAW_DIR, ARTIFACTS_DIR, OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

import torch

HAS_CUDA = torch.cuda.is_available()
HAS_MPS = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

DEVICE = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")

# VRAM-aware config: on 24GB+ GPUs we skip 4-bit quant and gradient checkpointing for ~30-40% speedup
VRAM_GB = 0.0
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1e9
    print(f"CUDA: {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GB VRAM)")
elif DEVICE == "mps":
    print("MPS detected.")
else:
    print("CPU only.")

LARGE_GPU = HAS_CUDA and VRAM_GB >= 24.0
USE_4BIT = HAS_CUDA and not LARGE_GPU
USE_GRAD_CKPT = not LARGE_GPU

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"HF token provided: {'yes' if HF_TOKEN else 'no'}")
print(f"python={platform.python_version()} torch={torch.__version__} device={DEVICE}")
print(f"large_gpu={LARGE_GPU} use_4bit={USE_4BIT} grad_ckpt={USE_GRAD_CKPT}")

CUDA: Tesla V100-PCIE-32GB (34.1 GB VRAM)
PROJECT_ROOT=/home/coder/workspace/financialbot/notebooks
HF token provided: no
python=3.11.13 torch=2.0.1+cu118 device=cuda
large_gpu=True use_4bit=False grad_ckpt=False


## Section 2 — Data processing and validation

This section converts raw collected samples into a clean training dataset suitable for fine-tuning.

### Expected folder layout

```
data/
  raw_chatgpt/
    planning_en.jsonl
    investment_vi.jsonl
    trading_zh.jsonl
    ...
  sft_finbot.jsonl          # cleaned training set
  sft_finbot.eval.jsonl     # held-out evaluation set (~10%)
```

### Minimum sample format (per record)

```json
{
  "profile": {
    "GOAL": "...",
    "INCOME_BAND": "<60K | 60-120K | 120K+",
    "CAPITAL_RANGE": "<10K | 10-50K | 50-250K | 250K+",
    "TIME_HORIZON": "SHORT (<1y) | MEDIUM (1-5y) | LONG (5y+)",
    "RISK_TOLERANCE": "LOW | MEDIUM | HIGH"
  },
  "unknown_fields": [],
  "response": {
    "internal_analysis": "...",
    "profile_summary": "...",
    "recommendation": "...",
    "reasoning": "...",
    "risks_caveats": "...",
    "sources": [],
    "disclaimer": "..."
  }
}
```

### What the parser does
- Reads both `*.jsonl` and `*.json`
- Infers missing `task_mode` / `language` from filenames (e.g., `planning_vi.jsonl`)
- Validates output with `RecommendationPayload`
- Removes unsafe, invalid, and duplicate samples

Output of this section should be a clear dataset-quality summary for your report.

In [4]:
import json, glob, hashlib
from pathlib import Path
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pydantic import ValidationError
from finbot.schemas import TaskMode, LanguageCode, RecommendationPayload
from finbot.prompt_builder import build_recommendation_messages

def profile_hash(sample: dict) -> str:
    key = json.dumps({
        "p": sample["profile"],
        "u": sorted(sample.get("unknown_fields", [])),
        "t": sample["task_mode"],
        "l": sample["language"],
    }, sort_keys=True, ensure_ascii=False)
    return hashlib.sha1(key.encode("utf-8")).hexdigest()

def safety_ok(resp_text: str) -> bool:
    low = resp_text.lower()
    return not any(t in low for t in SAFETY_UNSAFE_TERMS)

def infer_task_lang_from_filename(path: str) -> tuple[str | None, str | None]:
    # Expect names like: planning_en.jsonl
    stem = Path(path).stem
    parts = stem.split("_")
    if len(parts) < 2:
        return None, None
    task = VALID_TASK_MAP.get(parts[0].lower())
    lang = parts[1].lower()
    if lang not in VALID_LANG_CODES:
        return task, None
    return task, lang

def iter_samples(path: str):
    if path.endswith(".jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    continue
    else:
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        if isinstance(blob, dict):
            yield blob
        elif isinstance(blob, list):
            for item in blob:
                if isinstance(item, dict):
                    yield item

records = []
stats = {
    "files": 0,
    "raw": 0,
    "schema_fail": 0,
    "unsafe": 0,
    "dup": 0,
    "kept": 0,
    "missing_task_or_lang": 0,
}
seen = set()

all_paths = sorted(glob.glob(str(RAW_DIR / "*.jsonl"))) + sorted(glob.glob(str(RAW_DIR / "*.json")))

for path in all_paths:
    stats["files"] += 1
    file_task, file_lang = infer_task_lang_from_filename(path)
    for s in iter_samples(path):
        stats["raw"] += 1

        # Backfill task/language from filename if absent in record.
        task_mode = s.get("task_mode") or file_task
        language = s.get("language") or file_lang
        if not task_mode or not language:
            stats["missing_task_or_lang"] += 1
            continue
        task_mode = str(task_mode).upper()
        language = str(language).lower()
        
        resp = dict(s.get("response", {}))

        normalized = {
            "profile": s.get("profile", {}),
            "unknown_fields": s.get("unknown_fields", []),
            "task_mode": task_mode,
            "language": language,
            "response": resp,
        }

        try:
            payload = RecommendationPayload.model_validate(normalized["response"])
        except (ValidationError, KeyError, TypeError):
            stats["schema_fail"] += 1
            continue

        if not safety_ok(json.dumps(payload.model_dump(), ensure_ascii=False)):
            stats["unsafe"] += 1
            continue

        h = profile_hash(normalized)
        if h in seen:
            stats["dup"] += 1
            continue
        seen.add(h)

        records.append({**normalized, "response": payload.model_dump()})
        stats["kept"] += 1

print(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"Validated records: {len(records)}")

{
  "files": 9,
  "raw": 618,
  "schema_fail": 0,
  "unsafe": 2,
  "dup": 1,
  "kept": 615,
  "missing_task_or_lang": 0
}
Validated records: 615


### Comparison Table 1 — Dataset quality summary

Use this table in your report to show:
- class balance by task mode
- language coverage (EN/VI/ZH)
- preprocessing quality (kept vs removed records)

In [5]:
import pandas as pd

df = pd.DataFrame([
    {
        "task_mode": r["task_mode"],
        "language": r["language"],
        "profile_keys": len(r["profile"]),
        "unknown_count": len(r.get("unknown_fields", [])),
        "resp_chars": sum(len(str(v)) for v in r["response"].values()),
    }
    for r in records
])

if len(df):
    print("=== Per task_mode ===")
    print(df.groupby("task_mode").agg(
        count=("resp_chars", "size"),
        mean_resp_chars=("resp_chars", "mean"),
        mean_unknown=("unknown_count", "mean"),
    ).round(1))

    print("\n=== Per language ===")
    print(df.groupby("language").size().to_frame("count"))

    print("\n=== Pipeline yield ===")
    print(pd.Series(stats).to_frame("n"))
else:
    print("No records found. Drop raw JSON batches into data/raw_chatgpt/ first.")

=== Per task_mode ===
            count  mean_resp_chars  mean_unknown
task_mode                                       
INVESTMENT    196            894.4           0.4
PLANNING      210           1542.0           0.2
TRADING       209            628.1           0.2

=== Per language ===
          count
language       
en          208
vi          208
zh          199

=== Pipeline yield ===
                        n
files                   9
raw                   618
schema_fail             0
unsafe                  2
dup                     1
kept                  615
missing_task_or_lang    0


### Convert to training format + split data

This step reformats cleaned samples into chat-style records for SFT training.

Then it creates:
- training set (`sft_finbot.jsonl`)
- evaluation set (`sft_finbot.eval.jsonl`)

Use the printed counts in your report as evidence of train/eval design.

In [6]:
import random
from sklearn.model_selection import train_test_split

TRAIN_PATH = DATA_DIR / "sft_finbot.jsonl"
EVAL_PATH = DATA_DIR / "sft_finbot.eval.jsonl"

def to_sft_chat(rec: dict) -> dict:
    msgs = build_recommendation_messages(
        task_mode=TaskMode(rec["task_mode"]),
        lang_code=LanguageCode(rec["language"]),
        collected=rec["profile"],
        unknown_fields=rec.get("unknown_fields", []),
    )
    msgs.append({
        "role": "assistant",
        "content": json.dumps(rec["response"], ensure_ascii=False),
    })
    return {"messages": msgs}

if records:
    stratify_key = [f"{r['task_mode']}_{r['language']}" for r in records]
    train_records, eval_records = train_test_split(
        [to_sft_chat(r) for r in records],
        test_size=0.1,
        random_state=42,
        stratify=stratify_key,
    )

    with open(TRAIN_PATH, "w", encoding="utf-8") as f:
        for r in train_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(EVAL_PATH, "w", encoding="utf-8") as f:
        for r in eval_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"train={len(train_records)}  eval={len(eval_records)}")
    print(f"Saved -> {TRAIN_PATH}")
    print(f"Saved -> {EVAL_PATH}")
else:
    print("No records to split. Add raw data and rerun the collection cell.")

train=553  eval=62
Saved -> /home/coder/workspace/financialbot/notebooks/data/sft_finbot.jsonl
Saved -> /home/coder/workspace/financialbot/notebooks/data/sft_finbot.eval.jsonl


## Section 3 — Baseline evaluation (before fine-tuning)

Evaluate the original `Qwen/Qwen2.5-1.5B-Instruct` on the held-out set first. This provides a fair baseline for further comparison.

Key metrics:
- `schema_valid_rate`: structured-output reliability
- `safety_pass_rate`: policy/safety consistency
- `internal_analysis_present_rate`: reasoning field completeness
- `mean_output_tokens`, `mean_latency_ms`: efficiency indicators

In [7]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

_local_only = os.getenv("HF_LOCAL_FILES_ONLY", "0").strip().lower() in {"1", "true", "yes", "on"}
_load_kwargs = {"local_files_only": _local_only, **({"token": HF_TOKEN} if HF_TOKEN else {})}

def _load_eval():
    return [json.loads(l) for l in open(EVAL_PATH, "r", encoding="utf-8")]

def _dtype_for(dev):
    if dev == "cuda":
        return torch.bfloat16
    if dev == "mps":
        return torch.float16
    return torch.float32

def extract_first_json(text: str) -> str | None:
    depth = 0
    start = -1
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                return text[start:i + 1]
    return None

def score_response(text: str) -> dict:
    result = {"schema_ok": False, "safety_ok": safety_ok(text), "internal_analysis_ok": False}
    js = extract_first_json(text)
    if not js:
        return result
    try:
        p = RecommendationPayload.model_validate_json(js)
        result["schema_ok"] = True
        result["internal_analysis_ok"] = bool(p.internal_analysis and p.internal_analysis.strip())
    except ValidationError:
        pass
    return result

def evaluate(model, tokenizer, eval_records, max_new_tokens=GEN_MAX_NEW_TOKENS, label="model"):
    rows = []
    for i, rec in enumerate(eval_records):
        user_msg = rec["messages"][0]["content"]
        model_inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_msg}],
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
        input_len = model_inputs["input_ids"].shape[1]
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
        dt_ms = (time.perf_counter() - t0) * 1000
        gen_ids = out[0][input_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        s = score_response(text)
        rows.append({**s, "tokens_out": int(gen_ids.shape[0]), "latency_ms": dt_ms, "text": text})
    agg = {
        "label": label,
        "n": len(rows),
        "schema_valid_rate": sum(r["schema_ok"] for r in rows) / max(1, len(rows)),
        "safety_pass_rate": sum(r["safety_ok"] for r in rows) / max(1, len(rows)),
        "internal_analysis_present_rate": sum(r["internal_analysis_ok"] for r in rows) / max(1, len(rows)),
        "mean_output_tokens": sum(r["tokens_out"] for r in rows) / max(1, len(rows)),
        "mean_latency_ms": sum(r["latency_ms"] for r in rows) / max(1, len(rows)),
    }
    return agg, rows

t = time.perf_counter() # track progress
base_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, **_load_kwargs)
print(f"tokenizer load: {time.perf_counter()-t:.1f}s")    

if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token


t = time.perf_counter() # track progress
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=_dtype_for(DEVICE), trust_remote_code=True, **_load_kwargs,
).to(DEVICE)
print(f"model load+move: {time.perf_counter()-t:.1f}s")

base_model.eval()


eval_records = _load_eval()
baseline_agg, baseline_rows = evaluate(base_model, base_tok, eval_records, label="qwen1_5b_base")
with open(OUTPUT_DIR / "baseline_metrics.json", "w") as f:
    json.dump(baseline_agg, f, indent=2)
print(json.dumps(baseline_agg, indent=2))

tokenizer load: 0.7s
model load+move: 6.4s


/home/coder/workspace/financialbot/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/coder/workspace/financialbot/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/coder/workspace/financialbot/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


RuntimeError: "triu_tril_cuda_template" not implemented for 'BFloat16'

## Section 4 — Fine-tuning step (LoRA / QLoRA)

This section runs the actual adaptation of the base model. Config auto-adapts to GPU:
on **24 GB+ GPUs** (`LARGE_GPU=True`) we skip 4-bit quantization and gradient checkpointing
for ~30-40% faster training; on smaller GPUs (T4 16 GB, Colab free) we keep both for safety.

- `max_seq_length`: `2048` — holds prompt + JSON output without truncation.
- `per_device_train_batch_size`: `4`, `gradient_accumulation_steps`: `2` — effective batch 8.
- `packing=True` — concatenates short sequences into 2048-token chunks for ~2× throughput.
- `completion_only_loss=True` (when supported by TRL) — loss only on the assistant turn, so we don't waste capacity relearning the identical system prompt.
- `eval_strategy="no"` — full eval runs once in Section 5, not during training.
- `dataloader_num_workers=2` — parallel tokenization, small but free win.
- `attn_implementation="flash_attention_2"` — used automatically if `flash-attn` is installed on CUDA; silent fallback otherwise.
- `num_train_epochs`: `1` — assignment demo run; increase later for deeper tuning.
- `learning_rate`: `2e-4` — standard LoRA starting point.
- `lora_r`: `16`, `lora_alpha`: `32`, `lora_dropout`: `0.05` — good capacity/efficiency trade-off.
- `target_modules`: attention + MLP blocks — common Qwen LoRA targets.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

if "base_model" in globals():
    del base_model
if HAS_CUDA:
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, **_load_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_ds = load_dataset("json", data_files=str(TRAIN_PATH), split="train")
eval_ds = load_dataset("json", data_files=str(EVAL_PATH), split="train")

def format_chat(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = train_ds.map(format_chat, remove_columns=["messages"])
eval_ds = eval_ds.map(format_chat, remove_columns=["messages"])

# Prefer FlashAttention-2 on CUDA when available (~15-25% attn speedup). Silent fallback otherwise.
_attn_kwargs = {}
if HAS_CUDA:
    try:
        import flash_attn  # noqa: F401
        _attn_kwargs["attn_implementation"] = "flash_attention_2"
        print("Using FlashAttention-2")
    except Exception:
        print("flash_attn not available — using default attention")

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map="auto",
        trust_remote_code=True, **_load_kwargs, **_attn_kwargs,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=_dtype_for(DEVICE),
        trust_remote_code=True, **_load_kwargs, **_attn_kwargs,
    ).to(DEVICE)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

import inspect

sft_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=TRAIN_GRAD_ACC_STEPS,
    learning_rate=TRAIN_LR,
    num_train_epochs=TRAIN_EPOCHS,
    logging_steps=TRAIN_LOGGING_STEPS,
    save_strategy=TRAIN_SAVE_STRATEGY,
    bf16=HAS_CUDA,
    fp16=HAS_MPS,
    dataset_text_field="text",
    report_to=TRAIN_REPORT_TO,
    seed=TRAIN_SEED,
    gradient_checkpointing=USE_GRAD_CKPT,
    packing=TRAIN_PACKING,
    dataloader_num_workers=TRAIN_DATALOADER_WORKERS,
)

cfg_params = inspect.signature(SFTConfig).parameters
if "max_seq_length" in cfg_params:
    sft_kwargs["max_seq_length"] = TRAIN_MAX_SEQ_LENGTH
elif "max_length" in cfg_params:
    sft_kwargs["max_length"] = TRAIN_MAX_SEQ_LENGTH

# Skip mid-training eval (Section 5 runs full eval separately). Saves ~15-25% wall-clock.
if "eval_strategy" in cfg_params:
    sft_kwargs["eval_strategy"] = "no"
elif "evaluation_strategy" in cfg_params:
    sft_kwargs["evaluation_strategy"] = "no"

# Assistant-only loss masking — don't relearning the identical system prompt.
if "completion_only_loss" in cfg_params:
    sft_kwargs["completion_only_loss"] = True

sft_cfg = SFTConfig(**sft_kwargs)

trainer_kwargs = dict(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
)
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer

if "max_seq_length" in trainer_params:
    trainer_kwargs["max_seq_length"] = TRAIN_MAX_SEQ_LENGTH

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Adapter saved -> {OUTPUT_DIR}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

Adding EOS to eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.909036
20,0.473944
30,0.080160


Adapter saved -> /content/artifacts/lora-qwen25-1p5b-finbot


## Section 5 — Post-training evaluation

### Comparison Table 2 — Baseline vs fine-tuned

Evaluate the fine-tuned model on the same held-out set and same decoding setup (`do_sample=False`).

This is your main evidence that fine-tuning changed measurable performance.

In [ ]:
from peft import PeftModel

for _name in ("model", "trainer"):
    if _name in globals():
        del globals()[_name]
if HAS_CUDA:
    torch.cuda.empty_cache()

ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=_dtype_for(DEVICE), trust_remote_code=True, **_load_kwargs,
).to(DEVICE)
ft_model = PeftModel.from_pretrained(ft_base, str(OUTPUT_DIR))
ft_model.eval()

finetuned_agg, finetuned_rows = evaluate(ft_model, tokenizer, eval_records, label="qwen1_5b_finbot_lora")
with open(OUTPUT_DIR / "finetuned_metrics.json", "w") as f:
    json.dump(finetuned_agg, f, indent=2)

comp = pd.DataFrame([baseline_agg, finetuned_agg]).set_index("label")
print(comp.round(3))

for i in range(min(2, len(eval_records))):
    print("\n" + "=" * 80)
    print(f"EXAMPLE {i + 1}")
    print("-- BASE OUTPUT --\n", baseline_rows[i]["text"][:1200])
    print("-- FINETUNED OUTPUT --\n", finetuned_rows[i]["text"][:1200])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

                       n  schema_valid_rate  safety_pass_rate  \
label                                                           
qwen1_5b_base         53              0.000             1.000   
qwen1_5b_finbot_lora  53              0.057             0.962   

                      internal_analysis_present_rate  mean_output_tokens  \
label                                                                      
qwen1_5b_base                                  0.000             405.547   
qwen1_5b_finbot_lora                           0.057             304.736   

                      mean_latency_ms  
label                                  
qwen1_5b_base               19210.232  
qwen1_5b_finbot_lora        28471.417  

EXAMPLE 1
-- BASE OUTPUT --
 {
  "internal_analysis": {
    "capital_range": "50-250K",
    "income_band": "60-120K",
    "risk_tolerance": "MEDIUM",
    "time_horizon": "LONG (5y+)",
    "goal": "对冲现有大额美股持仓，防止未来五年汇率及宏观波动。",
    "unknown_fields": []
  },
  "profile_summary

## Section 6 — Before vs after fine-tuning

### Comparison Table 3 — Base vs fine-tuned (same model)

This section compares only `Qwen/Qwen2.5-1.5B-Instruct` under the same eval protocol:
- base model (`qwen1_5b_base`)
- fine-tuned LoRA adapter (`qwen1_5b_finbot_lora`)

Use this table to show the direct impact of fine-tuning on the same backbone model.

In [ ]:
if "ft_model" in globals():
    del ft_model
if "ft_base" in globals():
    del ft_base
if HAS_CUDA:
    torch.cuda.empty_cache()

cross_rows = [baseline_agg, finetuned_agg]
cross_df = pd.DataFrame(cross_rows).set_index("label")
print(cross_df.round(3))
with open(OUTPUT_DIR / "cross_model_metrics.json", "w") as f:
    json.dump(cross_rows, f, indent=2)

                       n  schema_valid_rate  safety_pass_rate  \
label                                                           
qwen1_5b_base         53              0.000             1.000   
qwen1_5b_finbot_lora  53              0.057             0.962   

                      internal_analysis_present_rate  mean_output_tokens  \
label                                                                      
qwen1_5b_base                                  0.000             405.547   
qwen1_5b_finbot_lora                           0.057             304.736   

                      mean_latency_ms  
label                                  
qwen1_5b_base               19210.232  
qwen1_5b_finbot_lora        28471.417  


## Section 7 — Multilingual performance check (EN / VI / ZH)

### Comparison Table 4 — Per-language results

This section reports baseline vs fine-tuned performance by language.

Use it to show multilingual capability and to identify language-specific regressions or gains.

In [ ]:
def _lang_of(rec_chat: dict) -> str:
    for lang in ("en", "vi", "zh"):
        if f"Language: {lang}" in rec_chat["messages"][0]["content"]:
            return lang
    return "unknown"

per_lang = [{"idx": i, "lang": _lang_of(r)} for i, r in enumerate(eval_records)]
lang_groups = {}
for p in per_lang:
    lang_groups.setdefault(p["lang"], []).append(p["idx"])

def _slice_agg(rows, idxs, label):
    sub = [rows[i] for i in idxs]
    return {
        "label": label,
        "n": len(sub),
        "schema_valid_rate": sum(r["schema_ok"] for r in sub) / max(1, len(sub)),
        "safety_pass_rate": sum(r["safety_ok"] for r in sub) / max(1, len(sub)),
        "internal_analysis_present_rate": sum(r["internal_analysis_ok"] for r in sub) / max(1, len(sub)),
    }

rows_out = []
for lang, idxs in lang_groups.items():
    rows_out.append(_slice_agg(baseline_rows, idxs, f"base_{lang}"))
    rows_out.append(_slice_agg(finetuned_rows, idxs, f"finetuned_{lang}"))

lang_df = pd.DataFrame(rows_out).set_index("label")
print(lang_df.round(3))
with open(OUTPUT_DIR / "cross_lingual_metrics.json", "w") as f:
    json.dump(rows_out, f, indent=2)

               n  schema_valid_rate  safety_pass_rate  \
label                                                   
base_zh       20              0.000              1.00   
finetuned_zh  20              0.000              1.00   
base_en       25              0.000              1.00   
finetuned_en  25              0.080              0.92   
base_vi        8              0.000              1.00   
finetuned_vi   8              0.125              1.00   

              internal_analysis_present_rate  
label                                         
base_zh                                0.000  
finetuned_zh                           0.000  
base_en                                0.000  
finetuned_en                           0.080  
base_vi                                0.000  
finetuned_vi                           0.125  


## Section 8 — Optional deployment note

This notebook already covers assignment evidence for training and evaluation.

If you later integrate the adapter into your application, load the base model first and then attach the LoRA adapter path. Example pattern:

```python
from peft import PeftModel

FINBOT_ADAPTER_DIR = os.getenv("FINBOT_ADAPTER_DIR")

@lru_cache(maxsize=3)
def _get_generator(model_id: str):
    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    if FINBOT_ADAPTER_DIR and model_id == "Qwen/Qwen2.5-1.5B-Instruct":
        model = PeftModel.from_pretrained(model, FINBOT_ADAPTER_DIR)
    return pipeline("text-generation", model=model, tokenizer=tokenizer)
```

For this assignment runbook, deployment is optional. The cell below only verifies that the saved adapter can generate schema-valid output.

In [ ]:
sanity_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=_dtype_for(DEVICE), trust_remote_code=True, **_load_kwargs,
).to(DEVICE)
sanity_model = PeftModel.from_pretrained(sanity_base, str(OUTPUT_DIR))
sanity_model.eval()

demo_profile = {
    "GOAL": "Grow a rainy-day fund and start retirement saving",
    "INCOME_BAND": "60-120K",
    "CAPITAL_RANGE": "10-50K",
    "TIME_HORIZON": "LONG (5y+)",
    "RISK_TOLERANCE": "MEDIUM",
}
demo_messages = build_recommendation_messages(
    task_mode=TaskMode.PLANNING,
    lang_code=LanguageCode.EN,
    collected=demo_profile,
    unknown_fields=[],
)
model_inputs = tokenizer.apply_chat_template(
    demo_messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
model_inputs = {k: v.to(sanity_model.device) for k, v in model_inputs.items()}
input_len = model_inputs["input_ids"].shape[1]
with torch.no_grad():
    out = sanity_model.generate(**model_inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False)
text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
js = extract_first_json(text) or "{}"
try:
    payload = RecommendationPayload.model_validate_json(js)
    print("Schema OK. Preview:")
    print(json.dumps(payload.model_dump(), indent=2, ensure_ascii=False)[:1200])
except ValidationError as e:
    print("Schema FAILED. Raw output below:")
    print(text[:1200])
    print("\nValidation error:", e)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Schema OK. Preview:
{
  "internal_analysis": "Because the user has a medium-risk tolerance and a long-term time horizon (>5 years), we recommend a balanced portfolio with a mix of fixed income and growth assets.",
  "profile_summary": "This individual falls into the mid-cap range with a solid net capability and a preference for stability.",
  "recommendation": "For a LONG term ($>5y) goal, we suggest establishing a 60% Fixed Income + 40% Growth allocation. This ensures adequate protection against inflation while providing potential for capital appreciation.",
  "reasoning": "A medium-risk tolerance allows for some volatility in pursuit of higher returns, but the user still wants to preserve capital. The 60/40 split balances these needs.",
  "risks_caveats": "If interest rates rise by 1%, the real return on fixed-income holdings may decline. Additionally, if the user fails to maintain the $X margin required for liquid funds, they could experience liquidity constraints.",
  "sources": []

## Section 9 — Reproducibility manifest

This final section exports one JSON artifact with:
- run timestamp
- dataset sizes and preprocessing stats
- training hyperparameters
- evaluation metrics
- package versions

Include this file in your appendix/report to support reproducibility and marking transparency.

In [ ]:
import datetime, transformers, peft, trl

manifest = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "base_model": MODEL_ID,
    "adapter_dir": str(OUTPUT_DIR),
    "dataset": {
        "train_path": str(TRAIN_PATH),
        "eval_path": str(EVAL_PATH),
        "train_size": len(train_ds),
        "eval_size": len(eval_ds),
        "pipeline_stats": stats,
    },
    "hyperparameters": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "learning_rate": TRAIN_LR, "num_train_epochs": TRAIN_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation_steps": TRAIN_GRAD_ACC_STEPS,
        "max_seq_length": TRAIN_MAX_SEQ_LENGTH, "seed": TRAIN_SEED,
        "packing": TRAIN_PACKING,
        "use_4bit": USE_4BIT, "device": DEVICE,
        "large_gpu": LARGE_GPU, "vram_gb": round(VRAM_GB, 1),
        "gradient_checkpointing": USE_GRAD_CKPT,
        "completion_only_loss": bool(sft_kwargs.get("completion_only_loss", False)),
        "flash_attention_2": _attn_kwargs.get("attn_implementation") == "flash_attention_2",
    },
    "metrics": {
        "baseline": baseline_agg,
        "finetuned": finetuned_agg,
        "cross_model": cross_rows,
        "cross_lingual": rows_out,
    },
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "trl": trl.__version__,
        "pandas": pd.__version__,
    },
}

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Manifest saved -> {manifest_path}")
print(json.dumps({"baseline": baseline_agg, "finetuned": finetuned_agg}, indent=2))

Manifest saved -> /content/artifacts/lora-qwen25-1p5b-finbot/manifest.json
{
  "baseline": {
    "label": "qwen1_5b_base",
    "n": 53,
    "schema_valid_rate": 0.0,
    "safety_pass_rate": 1.0,
    "internal_analysis_present_rate": 0.0,
    "mean_output_tokens": 405.54716981132077,
    "mean_latency_ms": 19210.23243373584
  },
  "finetuned": {
    "label": "qwen1_5b_finbot_lora",
    "n": 53,
    "schema_valid_rate": 0.05660377358490566,
    "safety_pass_rate": 0.9622641509433962,
    "internal_analysis_present_rate": 0.05660377358490566,
    "mean_output_tokens": 304.7358490566038,
    "mean_latency_ms": 28471.417492490596
  }
}


/tmp/ipykernel_519/2743435683.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat(),
